# Part 1 — MultiModal RAG over a Read-Only Knowledge Base

**Ingest** (PyMuPDF): text passages; numbered tables as structured text; every numbered figure and table cropped to an image, with captions matched by page geometry. Figures are made searchable by VLM text descriptions (task 1.1.3).

**Index**: ChromaDB (bge-small dense) + BM25, fused with RRF, reranked by a cross-encoder. `doc_id`, `page`, `figure_id`, `trust_level` come from `kb_manifest.json`; low-trust content is filtered before ranking.

**Answer**, by question type:
- *Plain data lookup*: table cells become atomic facts keyed by (entity, variable, year). Comparing values under one key applies the manifest's precedence (revision, outturn over forecast, conflict) without the model.
- *Named figure*: a focused VLM call on exactly that chart, so near-identical figures cannot be mixed.
- *Explanation*: the best passages and an explanation prompt.
- *Calculation or assumption*: matching cells, a checked-lookup note and a reasoning prompt. User values stay in that conversation's memory.

Citations and figure references are built from the source ids the model selects, so they cannot be malformed or invented.

In [ ]:
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
    text=True).stdout.strip()
import glob
if not commit:  # on Kaggle there is no git checkout: the code dataset carries the commit it was built from
    found = glob.glob("/kaggle/input/**/part1/member-nikhil/COMMIT", recursive=True)
    commit = open(found[0]).read().strip() if found else "unknown"
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})


In [ ]:
import os, sys, subprocess
# Find the code: next to this notebook in the repo, or in the attached Kaggle code dataset.
REPO_MEMBER_DIR = os.path.abspath("..")
if not os.path.isdir(os.path.join(REPO_MEMBER_DIR, "src", "part1")):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "kaggle_submit.py" in files:
            REPO_MEMBER_DIR = os.path.dirname(os.path.dirname(root))
            break
print("code:", REPO_MEMBER_DIR)
if os.path.isdir("/kaggle/working"):
    # Versions the pipeline was developed against; Kaggle's preinstalled torch is kept.
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                          os.path.join(REPO_MEMBER_DIR, "kaggle_requirements.txt")], capture_output=True, text=True)
    print(pip.stdout[-1500:], pip.stderr[-1500:])
    assert pip.returncode == 0, "dependency install failed"
sys.path.insert(0, REPO_MEMBER_DIR)
os.chdir(REPO_MEMBER_DIR)   # scripts expect the member folder as cwd
from src.part1.config import load_config
config = load_config()
print(config)


## 1.1–1.3 Ingest, index, answer every conversation turn by turn (own memory per `conversation_id`)
Graders swap `QUESTIONS_PATH` for the hidden set.

In [ ]:
import torch
from src.part1.kaggle_submit import main
submission = main()   # ingest -> index -> answer -> write submission.csv -> run validate_submission.py
print("peak GPU GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else "n/a")


In [ ]:
import json
work = "/kaggle/working" if os.path.isdir("/kaggle/working") else "data/processed/part1"
print(json.dumps(json.load(open(os.path.join(work, "run_manifest.json"))), indent=2))
submission[["status", "figure_references", "citations", "answer"]].head(10)


## 1.2 Ingestion check: every numbered figure with its exact label

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from PIL import Image
paths = sorted(p for p in glob.glob(os.path.join(work, "figures", "*.png")) if "Figure" in os.path.basename(p))
print(len(glob.glob(os.path.join(work, "figures", "*.png"))), "objects cropped;", len(paths), "are figures")
rows = (len(paths) + 4) // 5
fig, axes = plt.subplots(rows, 5, figsize=(20, 3.2 * rows))
for ax in axes.flat:
    ax.axis("off")
for ax, path in zip(axes.flat, paths):
    ax.imshow(Image.open(path)); ax.set_title(os.path.basename(path).replace(".png", ""), fontsize=8)
plt.tight_layout(); plt.show()


## 1.4 Evaluation on the 25-question golden set
Three configurations, same index and model: **baseline** (dense retrieval only), **no reranker** (the final pipeline with the cross-encoder switched off) and **final** (hybrid + reranker). The no-reranker row isolates the reranker's contribution (bonus S.5).

In [ ]:
import pandas as pd
from src.part1.kaggle_submit import find_data_dir, ensure_index
from src.part1.score_golden_set import run_golden_set, write_rating_sheet
index = ensure_index(find_data_dir(), work)
runs = {"baseline (dense only)": run_golden_set(index, baseline=True),
        "hybrid, no reranker": run_golden_set(index, rerank=False),
        "final (hybrid + reranker)": run_golden_set(index)}
final = runs["final (hybrid + reranker)"]
# Saved with the run outputs so the comparison can be read back without the notebook display.
with open(os.path.join(work, "golden_eval.json"), "w") as f:
    json.dump({name: dict({k: v for k, v in r.items() if k != "details"},
                          items={d["id"]: {"correct": d["correct"], "recall@5": d["recall_hit"],
                                           "status": d["predicted"]["status"], "answer": d["predicted"]["answer"]}
                                 for d in r["details"]})
               for name, r in runs.items()}, f, indent=1)
comparison = pd.DataFrame({name: {k: v for k, v in r.items() if k != "details"} for name, r in runs.items()}).T
print(comparison.to_string())
comparison

In [ ]:
# Answers that changed when the reranker was switched on
off = {d["id"]: d for d in runs["hybrid, no reranker"]["details"]}
pd.DataFrame([{"id": d["id"], "correct without": off[d["id"]]["correct"], "correct with": d["correct"],
               "answer without": off[d["id"]]["predicted"]["answer"], "answer with": d["predicted"]["answer"]}
              for d in final["details"] if d["correct"] != off[d["id"]]["correct"]])

### Human evaluation sheet
One row per final answer with the text of every cited source. Both members fill `rater_nikhil` / `rater_anushka` (1-5 groundedness) independently; `summarize_ratings` reports the mean and agreement within one point.

In [ ]:
sheet = os.path.join(work, "part1_groundedness_sheet.csv")
write_rating_sheet(final, sheet)
print("wrote", sheet)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.DataFrame([{"id": d["id"], "correct": d["correct"], "recall@5": d["recall_hit"], "expected": d["expected_status"],
               "status": d["predicted"]["status"], "answer": d["predicted"]["answer"]} for d in final["details"]])
